In [11]:
import pandas as pd
import csv
import pickle
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import os
import random
from itertools import product
import pandas as pd
import statsmodels.api as sm


from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search

In [12]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)

In [13]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [14]:
random.seed(42)
# create feature matrix
X = feature_matrix.copy()
random.seed(42)

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# create featrue matrix containing only stocks
y = feature_matrix[stock_cols]
X = feature_matrix[topic_cols]

y = y.sample(n=10, axis=1, random_state=42)

# get innovations for X
X = to_ar1_innovations(X)

# remove the first row wiht iloc
X = X.iloc[1:]

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

In [15]:
import pandas as pd
import numpy as np
from grid_search import estimate_single_config

def run_all_y_columns(
    X_stocks, X_topics, window_size=350, n_lags=8, lambda_val=0.0021784
):
    all_summaries = []
    all_details = []

    for col_name in X_stocks.columns:
        print(f"Running column {col_name}...")

        y = X_stocks[col_name]

        result = estimate_single_config(
            X=X_topics,
            y=y,
            window_size=window_size,
            n_lags=n_lags,
            lambda_val=lambda_val
        )

        # --- summary ---
        summary = pd.DataFrame([result["summary"]])
        summary["target_column"] = col_name
        all_summaries.append(summary)

        # --- details ---
        details = result["details"].copy()
        details["target_column"] = col_name
        all_details.append(details)

    summaries_df = pd.concat(all_summaries, ignore_index=True)
    details_df = pd.concat(all_details, ignore_index=True)

    return summaries_df, details_df


In [16]:
summaries, details = run_all_y_columns(y, X)

print("All Summaries:")
print(summaries.head())

print("\nAll Details:")
print(details.head())

Running column 23393...
Running column 67360...
Running column 92635...
Running column 75341...
Running column 92951...
Running column 89179...
Running column 92949...
Running column 80206...
Running column 83976...
Running column 43123...
All Summaries:
   window_size  n_lags    lambda  r2_insample_stage1  r2_oos_stage1  \
0          350       8  0.002178            0.001883      -0.005641   
1          350       8  0.002178            0.010245      -0.004135   
2          350       8  0.002178            0.220369      -0.053369   
3          350       8  0.002178            0.072856      -0.034114   
4          350       8  0.002178            0.051321      -0.005407   

   r2_insample_stage2  r2_oos_stage2     kappa  kappa_tstat  intercept  \
0            0.001628       0.000795  0.676020     4.874177   0.000748   
1            0.000065      -0.000039  0.136932     0.364054   0.000657   
2            0.001642      -0.003242  0.103065     1.765618   0.000389   
3            0.000386 

C:\Users\jonat\AppData\Local\Temp\ipykernel_31032\1143336549.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  details_df = pd.concat(all_details, ignore_index=True)


In [17]:
pd.DataFrame.to_parquet(summaries, 'stage1_stage2_summaries.parquet')
pd.DataFrame.to_parquet(details, 'stage1_stage2_details.parquet')